# Лабораторная работа № 1
Есиков Сергей 

СПбАУ, 302 гр.

Вар 5

### Задание
![alt text](../tasks/1.png)

In [25]:
import pandas as pd
import numpy as np

In [26]:
MY_VARIANT = 5
df = pd.read_csv('../data/1.csv', header=None, names=['variant', 'X1', 'X2'])

## 1. Выбрать данные своего варианта

In [27]:
data = df[df['variant'] == MY_VARIANT][['X1', 'X2']].copy()
n = len(data)
print(f"Количество наблюдений в варианте {MY_VARIANT}: {n}")
data.head(5)

Количество наблюдений в варианте 5: 977


,X1,X2
4000,92.694,198.508
4001,86.209,197.678
4002,89.561,195.832
4003,92.763,201.730
4004,91.663,203.199


## 2. Разбить наблюдения на 10 групп примерно одного размера

In [28]:
data_shuffled = data.sample(frac=1)
groups = [pd.DataFrame(arr, columns=data_shuffled.columns) for arr in np.array_split(data_shuffled, 10)]

## 3. Для каждой из групп найти выборочные средние и медиану для каждого из наблюдений.

In [29]:
def calc_stats(group):
    return {
        'X1_mean': group['X1'].mean(),
        'X1_median': group['X1'].median(),
        'X2_mean': group['X2'].mean(),
        'X2_median': group['X2'].median()
    }

stats_list = [calc_stats(g) for g in groups]
stats_df = pd.DataFrame(stats_list)


In [30]:
stats_df

,X1_mean,X1_median,X2_mean,X2_median
0,90.920061,90.9635,208.752000,202.8560
1,90.950316,90.6955,201.304990,202.4115
2,90.855776,90.9580,193.889694,200.6675
3,91.243429,90.9175,202.544684,202.5550
4,91.036878,91.0670,202.970122,202.0055
5,91.358745,91.0635,133.541490,201.6885
6,91.185276,91.3360,188.460561,202.4670
7,90.246619,90.5080,206.119557,200.6330
8,91.522268,91.5260,204.459093,203.1880
9,90.889237,91.1550,208.753794,201.6230


## 4. Получится 10 оценок медианы и 10 оценок среднего. Считая их элементами выборки объема 10, проанализировать, какая из них имеет меньший разброс.

In [31]:
for var in ['X1', 'X2']:
    mean_std = stats_df[f'{var}_mean'].std(ddof=1)
    median_std = stats_df[f'{var}_median'].std(ddof=1)
    print(f"{var}: std(среднее) = {mean_std:.4f}, std(медиана) = {median_std:.4f}")
    print(f"{"медиана" if median_std < mean_std else "среднее"} имеет разброс меньше")

X1: std(среднее) = 0.3499, std(медиана) = 0.2912
медиана имеет разброс меньше
X2: std(среднее) = 22.5405, std(медиана) = 0.8642
медиана имеет разброс меньше


## 5. Проделать то же самое для выборочных квантилей на уровнях 0.2, 0.4, 0.6, 0.8.

In [32]:
quantiles = [0.2, 0.4, 0.6, 0.8]
quantile_dict = {}

for var in ['X1', 'X2']:
    for q in quantiles:
        col_name = f'{var}_q{q}'
        stats_df[col_name] = [g[var].quantile(q) for g in groups]

for var in ['X1', 'X2']:
    print(f"\n{var}:")
    base_stats = ['mean', 'median']
    for s in base_stats:
        std_val = stats_df[f'{var}_{s}'].std(ddof=1)
        print(f"  std({s}) = {std_val:.4f}")
    for q in quantiles:
        col = f'{var}_q{q}'
        std_val = stats_df[col].std(ddof=1)
        print(f"  std(q={q}) = {std_val:.4f}")


X1:
  std(mean) = 0.3499
  std(median) = 0.2912
  std(q=0.2) = 0.3642
  std(q=0.4) = 0.3284
  std(q=0.6) = 0.3644
  std(q=0.8) = 0.4988

X2:
  std(mean) = 22.5405
  std(median) = 0.8642
  std(q=0.2) = 2.3982
  std(q=0.4) = 1.1298
  std(q=0.6) = 0.9537
  std(q=0.8) = 0.8237


## 6. Продумать, какие из этого можно сделать выводы.

1. Стандартная ошибка среднего X1 меньше ошибки медианы, при чём не сильно, что может говорить об отсутствии у распределения тяжёлых хвостов
2. Стандартные отклонения симметричных квантилей X1 сильно различаются, что может говорить о смещении плотности распределения в правую сторону
3. Стандартная ошибка среднего для X2 огромна, в то время как ошибка медианы минимальна, чьл свидетельствует о наличи выбросов или тяжёлых хвостов
4. Симметрично расположенные квантили - показатель симметрии самого распределения